In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/SONY_daily_data.csv
/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/DELL_daily_data.csv
/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/MSFT_daily_data.csv
/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/VZ_daily_data.csv
/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/IBM_daily_data.csv
/kaggle/input/microsoft-stock-data-and-key-affiliated-companies/INTC_daily_data.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import sqlite3
import re
import os

class StockMarketDatabase:
    def __init__(self, db_name=':memory:'):
        self.conn = sqlite3.connect(db_name)
        self.cursor = self.conn.cursor()
    
    def create_tables(self):
        self.cursor.execute('''
        CREATE TABLE IF NOT EXISTS stock_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            company TEXT NOT NULL,
            Date DATE NOT NULL,
            Open REAL,
            High REAL,
            Low REAL,
            Close REAL,
            [Adj Close] REAL,
            Volume INTEGER
        )
        ''')
        self.conn.commit()
    
    def load_data_from_csv(self, csv_files):
        for company, file_path in csv_files.items():
            try:
                print(f"Attempting to load data for {company} from {file_path}")
                df = pd.read_csv(file_path)
                df['company'] = company
                df.to_sql('stock_data', self.conn, if_exists='append', index=False)
                print(f"Successfully loaded data for {company}")
            except FileNotFoundError:
                print(f"File not found: {file_path}")
            except Exception as e:
                print(f"Error loading data for {company}: {str(e)}")
    
    def execute_query(self, query):
        try:
            self.cursor.execute(query)
            return self.cursor.fetchall()
        except sqlite3.Error as e:
            return f"Error executing query: {e}"

class TextToSQLConverter:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=1000)
        self.label_encoder = LabelEncoder()
        self.model = None
    
    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def create_sample_data(self):
        natural_queries = [
            "Show the latest stock price for Microsoft",
            "What was the highest price for Intel in 2023?",
            "Compare the closing prices of Microsoft and IBM for the last 30 days",
            "Which company had the highest trading volume last week?",
            "Calculate the average closing price for Dell in 2022",
            "Show the days when Sony's stock price increased by more than 5%",
            "What was the total trading volume for Microsoft in January 2024?",
            "Find the date when IBM had its lowest stock price in the last 5 years",
            "Compare the performance of all companies over the last quarter",
            "What is the price difference between Microsoft's highest and lowest prices in 2023?",
            "Show the top 5 days with the highest trading volume for Microsoft",
            "Calculate the monthly average closing price for all companies in 2023",
            "Which company had the most volatile stock price in the last year?",
            "Find weeks where Dell outperformed Microsoft",
            "Show the correlation between Microsoft and Intel stock prices",
            "What was the total trading volume for Sony in the first quarter of 2023?",
            "Show me the average closing price for Intel in 2023",
            "Compare the daily trading volumes of IBM and Dell for the past week",
            "Which company had the highest stock price increase in percentage over the last year?",
            "On which date did Dell have its lowest closing price in the dataset?"
        ]
        
        sql_queries = [
            "SELECT * FROM stock_data WHERE company = 'MSFT' ORDER BY Date DESC LIMIT 1",
            "SELECT MAX(High) FROM stock_data WHERE company = 'INTC' AND strftime('%Y', Date) = '2023'",
            "SELECT Date, MSFT.Close as MSFT_close, IBM.Close as IBM_close FROM (SELECT Date, Close FROM stock_data WHERE company = 'MSFT' ORDER BY Date DESC LIMIT 30) as MSFT JOIN (SELECT Date, Close FROM stock_data WHERE company = 'IBM' ORDER BY Date DESC LIMIT 30) as IBM ON MSFT.Date = IBM.Date ORDER BY Date",
            "SELECT company, SUM(Volume) as total_volume FROM stock_data WHERE Date >= date('now', '-7 days') GROUP BY company ORDER BY total_volume DESC LIMIT 1",
            "SELECT AVG(Close) FROM stock_data WHERE company = 'DELL' AND strftime('%Y', Date) = '2022'",
            "SELECT Date, (Close - Open) / Open * 100 as price_increase FROM stock_data WHERE company = 'SONY' AND (Close - Open) / Open > 0.05 ORDER BY Date",
            "SELECT SUM(Volume) FROM stock_data WHERE company = 'MSFT' AND strftime('%Y-%m', Date) = '2024-01'",
            "SELECT Date, Low FROM stock_data WHERE company = 'IBM' AND Date >= date('now', '-5 years') ORDER BY Low ASC LIMIT 1",
            "SELECT company, AVG(Close) as avg_close FROM stock_data WHERE Date >= date('now', '-3 months') GROUP BY company ORDER BY avg_close DESC",
            "SELECT MAX(High) - MIN(Low) as price_difference FROM stock_data WHERE company = 'MSFT' AND strftime('%Y', Date) = '2023'",
            "SELECT Date, Volume FROM stock_data WHERE company = 'MSFT' ORDER BY Volume DESC LIMIT 5",
            "SELECT company, strftime('%Y-%m', Date) as month, AVG(Close) as avg_close FROM stock_data WHERE strftime('%Y', Date) = '2023' GROUP BY company, month ORDER BY company, month",
            "SELECT company, (MAX(High) - MIN(Low)) / AVG(Close) as volatility FROM stock_data WHERE Date >= date('now', '-1 year') GROUP BY company ORDER BY volatility DESC LIMIT 1",
            "SELECT DELL.Date, DELL.Close as DELL_close, MSFT.Close as MSFT_close FROM stock_data DELL JOIN stock_data MSFT ON DELL.Date = MSFT.Date WHERE DELL.company = 'DELL' AND MSFT.company = 'MSFT' AND DELL.Close > MSFT.Close AND DELL.Date >= date('now', '-1 year') GROUP BY strftime('%W', DELL.Date) ORDER BY DELL.Date",
            "SELECT CORR(MSFT.Close, INTC.Close) as correlation FROM (SELECT Date, Close FROM stock_data WHERE company = 'MSFT') as MSFT JOIN (SELECT Date, Close FROM stock_data WHERE company = 'INTC') as INTC ON MSFT.Date = INTC.Date",
            "SELECT SUM(Volume) FROM stock_data WHERE company = 'SONY' AND strftime('%Y', Date) = '2023' AND strftime('%m', Date) BETWEEN '01' AND '03'",
            "SELECT AVG(Close) FROM stock_data WHERE company = 'INTC' AND strftime('%Y', Date) = '2023'",
            "SELECT Date, IBM.Volume as IBM_volume, DELL.Volume as DELL_volume FROM (SELECT Date, Volume FROM stock_data WHERE company = 'IBM' AND Date >= date('now', '-7 days')) as IBM JOIN (SELECT Date, Volume FROM stock_data WHERE company = 'DELL' AND Date >= date('now', '-7 days')) as DELL ON IBM.Date = DELL.Date ORDER BY Date",
            "SELECT company, (MAX(Close) - MIN(Close)) / MIN(Close) * 100 as price_increase FROM stock_data WHERE Date >= date('now', '-1 year') GROUP BY company ORDER BY price_increase DESC LIMIT 1",
            "SELECT Date, Close FROM stock_data WHERE company = 'DELL' ORDER BY Close ASC LIMIT 1"
        ]
        
        return natural_queries, sql_queries
    
    def prepare_data(self):
        natural_queries, sql_queries = self.create_sample_data()
        processed_queries = [self.preprocess_text(query) for query in natural_queries]
        X = self.vectorizer.fit_transform(processed_queries).toarray()
        y = self.label_encoder.fit_transform(sql_queries)
        y = to_categorical(y)
        return train_test_split(X, y, test_size=0.2, random_state=42)
    
    def build_model(self, input_shape, output_shape):
        model = Sequential([
            Dense(512, activation='relu', input_shape=(input_shape,)),
            Dropout(0.3),
            Dense(256, activation='relu'),
            Dropout(0.3),
            Dense(128, activation='relu'),
            Dropout(0.3),
            Dense(output_shape, activation='softmax')
        ])
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
        return model
    
    def train(self):
        X_train, X_test, y_train, y_test = self.prepare_data()
        self.model = self.build_model(X_train.shape[1], y_train.shape[1])
        return self.model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), verbose=1)
    
    def predict(self, natural_query):
        # First, try to match the query to predefined patterns
        sql_query = self.rule_based_prediction(natural_query)
        if sql_query:
            return sql_query
        
        # If no rule matches, use the neural network
        processed_query = self.preprocess_text(natural_query)
        query_vector = self.vectorizer.transform([processed_query]).toarray()
        prediction = self.model.predict(query_vector)
        predicted_index = np.argmax(prediction)
        return self.label_encoder.inverse_transform([predicted_index])[0]
    def rule_based_prediction(self, query):
        query = query.lower()
        
        # Pattern for total trading volume for a specific company in a specific quarter
        volume_pattern = r"total trading volume for (\w+) in the (first|second|third|fourth) quarter of (\d{4})"
        volume_match = re.search(volume_pattern, query)
        if volume_match:
            company, quarter, year = volume_match.groups()
            quarter_map = {"first": "01", "second": "04", "third": "07", "fourth": "10"}
            start_month = quarter_map[quarter]
            end_month = str(int(start_month) + 2).zfill(2)
            return f"""
                SELECT SUM(Volume) 
                FROM stock_data 
                WHERE company = '{company.upper()}' 
                AND strftime('%Y', Date) = '{year}' 
                AND strftime('%m', Date) BETWEEN '{start_month}' AND '{end_month}'
            """
        
        # Add more patterns here for other types of queries
        
        return None  # If no pattern matches, return None


def main():
    current_dir = os.getcwd()
    print(f"Current working directory: {current_dir}")
    
    default_csv_files = {
        'MSFT': 'MSFT.csv',
        'INTC': 'INTC.csv',
        'IBM': 'IBM.csv',
        'DELL': 'DELL.csv',
        'SONY': 'SONY.csv'
    }
    
    csv_dir = input("Enter the directory containing CSV files (press Enter to use current directory): ").strip()
    
    csv_files = {}
    for company, default_filename in default_csv_files.items():
        while True:
            file_path = input(f"Enter the path for {company} CSV file (or press Enter to use {os.path.join(csv_dir, default_filename)}): ").strip()
            if not file_path:
                file_path = os.path.join(csv_dir, default_filename)
            
            if os.path.exists(file_path):
                csv_files[company] = file_path
                break
            else:
                print(f"File not found: {file_path}")
                retry = input("Do you want to try again? (yes/no): ").lower()
                if retry != 'yes':
                    print(f"Skipping {company}")
                    break
    
    print("Initializing database...")
    try:
        db = StockMarketDatabase()
        db.create_tables()
        db.load_data_from_csv(csv_files)
        
        if not csv_files:
            print("No valid CSV files were provided. Exiting the program.")
            return
        
        print("Training Text-to-SQL model...")
        converter = TextToSQLConverter()
        converter.train()
        
        print("Ready to process queries!")
        while True:
            user_input = input("Enter your question (or 'quit' to exit): ")
            
            if user_input.lower() == 'quit':
                break
            
            sql_query = converter.predict(user_input)
            print(f"Generated SQL: {sql_query}")
            
            results = db.execute_query(sql_query)
            print("Results:", results)
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Current working directory: /kaggle/working


Enter the directory containing CSV files (press Enter to use current directory):  /kaggle/input/microsoft-stock-data-and-key-affiliated-companies
Enter the path for MSFT CSV file (or press Enter to use /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/MSFT.csv):  /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/MSFT_daily_data.csv
Enter the path for INTC CSV file (or press Enter to use /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/INTC.csv):  /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/INTC_daily_data.csv
Enter the path for IBM CSV file (or press Enter to use /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/IBM.csv):  /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/IBM_daily_data.csv
Enter the path for DELL CSV file (or press Enter to use /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/DELL.csv):  /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/DELL_daily_da

Initializing database...
Attempting to load data for MSFT from /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/MSFT_daily_data.csv
Successfully loaded data for MSFT
Attempting to load data for INTC from /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/INTC_daily_data.csv
Successfully loaded data for INTC
Attempting to load data for IBM from /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/IBM_daily_data.csv
Successfully loaded data for IBM
Attempting to load data for DELL from /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/DELL_daily_data.csv
Successfully loaded data for DELL
Attempting to load data for SONY from /kaggle/input/microsoft-stock-data-and-key-affiliated-companies/SONY_daily_data.csv
Successfully loaded data for SONY
Training Text-to-SQL model...
Epoch 1/100


/opt/conda/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 2.9980 - val_accuracy: 0.0000e+00 - val_loss: 3.0102
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.0000e+00 - loss: 2.9848 - val_accuracy: 0.0000e+00 - val_loss: 3.0133
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.2500 - loss: 2.9367 - val_accuracy: 0.0000e+00 - val_loss: 3.0172
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.3750 - loss: 2.9321 - val_accuracy: 0.0000e+00 - val_loss: 3.0209
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.1250 - loss: 2.9215 - val_accuracy: 0.0000e+00 - val_loss: 3.0267
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5625 - loss: 2.8864 - val_accuracy: 0.0000e+00 - val_loss: 3.0344
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5000 - loss: 2.8707 - val_accuracy: 0.0000e+00 - val_loss: 3.0456
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5625 - loss: 2.8207 - val_accur

Enter your question (or 'quit' to exit):   What was the total trading volume for Sony in the first quarter of 2023?


Generated SQL: 
                SELECT SUM(Volume) 
                FROM stock_data 
                WHERE company = 'SONY' 
                AND strftime('%Y', Date) = '2023' 
                AND strftime('%m', Date) BETWEEN '01' AND '03'
            
Results: [(48628400,)]
